In [57]:
# EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ML
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier

### Carga de Dados

In [58]:
df_leads = pd.read_csv('./datasets/leads.csv')

In [97]:
df_leads.info()

<class 'pandas.DataFrame'>
Index: 9074 entries, 0 to 9239
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Lead Origin                             9074 non-null   str    
 1   Lead Source                             9074 non-null   str    
 2   Do Not Email                            9074 non-null   int64  
 3   Do Not Call                             9074 non-null   int64  
 4   Converted                               9074 non-null   int64  
 5   TotalVisits                             9074 non-null   float64
 6   Total Time Spent on Website             9074 non-null   int64  
 7   Page Views Per Visit                    9074 non-null   float64
 8   Last Activity                           9074 non-null   str    
 9   Search                                  9074 non-null   int64  
 10  Newspaper Article                       9074 non-null   int64  
 11  X Educa

In [96]:
df_leads.head(10)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
0,API,Olark Chat,0,0,0,0.0,0,0.0,Page Visited on Website,0,0,0,0,0,0,0,Modified
1,API,Organic Search,0,0,0,5.0,674,2.5,Email Opened,0,0,0,0,0,0,0,Email Opened
2,Landing Page Submission,Direct Traffic,0,0,1,2.0,1532,2.0,Email Opened,0,0,0,0,0,0,1,Email Opened
3,Landing Page Submission,Direct Traffic,0,0,0,1.0,305,1.0,Unreachable,0,0,0,0,0,0,0,Modified
4,Landing Page Submission,Google,0,0,1,2.0,1428,1.0,Converted to Lead,0,0,0,0,0,0,0,Modified
5,API,Olark Chat,0,0,0,0.0,0,0.0,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
6,Landing Page Submission,Google,0,0,1,2.0,1640,2.0,Email Opened,0,0,0,0,0,0,0,Modified
7,API,Olark Chat,0,0,0,0.0,0,0.0,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
8,Landing Page Submission,Direct Traffic,0,0,0,2.0,71,2.0,Email Opened,0,0,0,0,0,0,1,Email Opened
9,API,Google,0,0,0,4.0,58,4.0,Email Opened,0,0,0,0,0,0,0,Email Opened


In [95]:
df_leads.tail(10)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
9230,Landing Page Submission,Google,0,0,0,2.0,870,2.00,Email Opened,0,0,0,0,0,0,0,Email Opened
9231,Landing Page Submission,Google,0,0,1,8.0,1016,4.00,Email Opened,0,0,0,0,0,0,0,Email Opened
9232,Landing Page Submission,Direct Traffic,0,0,0,2.0,1770,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9233,API,Direct Traffic,0,0,1,13.0,1409,2.60,SMS Sent,0,0,0,0,0,0,0,SMS Sent
9234,Landing Page Submission,Direct Traffic,0,0,1,5.0,210,2.50,SMS Sent,0,0,0,0,0,0,0,Modified
9235,Landing Page Submission,Direct Traffic,1,0,1,8.0,1845,2.67,Email Marked Spam,0,0,0,0,0,0,0,Email Marked Spam
9236,Landing Page Submission,Direct Traffic,0,0,0,2.0,238,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9237,Landing Page Submission,Direct Traffic,1,0,0,2.0,199,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9238,Landing Page Submission,Google,0,0,1,3.0,499,3.00,SMS Sent,0,0,0,0,0,0,0,SMS Sent
9239,Landing Page Submission,Direct Traffic,0,0,1,6.0,1279,3.00,SMS Sent,0,0,0,0,0,0,1,Modified


### Engenharia de Dados

In [62]:
df_leads.drop(['Prospect ID', 'Lead Number'], axis=1, inplace=True)

In [63]:
# Remove colunas com apenas um valor único

for column in df_leads.select_dtypes('object'):
    if df_leads[column].nunique() == 1:
        print(f"{column}: {df_leads[column].unique}")

        df_leads.drop(column, axis=1, inplace=True)

Magazine: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Magazine, Length: 9240, dtype: str>
Receive More Updates About Our Courses: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Receive More Updates About Our Courses, Length: 9240, dtype: str>
Update me on Supply Chain Content: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Update me on Supply Chain Content, Length: 9240, dtype: str>
Get updates on DM Content: <bound method Series.unique of 0       No
1       No
2       No
3       No
4       No
        ..
9235    No
9236    No
9237    No
9238    No
9239    No
Name: Get updates on DM Content, Length: 9240, dtype: str>
I agree to pay the amount through cheque

/tmp/ipykernel_15765/3997046926.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes('object'):


In [69]:
for column in df_leads.select_dtypes(include='object'):
    contagem_nulas = (df_leads[column] == 'Select').sum() + df_leads[column].isnull().sum()

    if contagem_nulas / len(df_leads) * 100 > 25:
        print(f'{column}: {contagem_nulas / len(df_leads) * 100:.2f}%')

        df_leads.drop(column, axis=1, inplace=True)

Country: 26.63%
Specialization: 36.58%
How did you hear about X Education: 78.46%
What is your current occupation: 29.11%
What matters most to you in choosing a course: 29.32%
Tags: 36.29%
Lead Quality: 51.59%
Lead Profile: 74.19%
City: 39.71%
Asymmetrique Activity Index: 45.65%
Asymmetrique Profile Index: 45.65%


/tmp/ipykernel_15765/4101281906.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes(include='object'):


In [79]:
df_leads['Lead Source'] = df_leads['Lead Source'].apply(lambda x: 'Google' if x == 'google' else x)

In [84]:
for column in df_leads.select_dtypes(include='object'):
    valores_unicos = df_leads[column].unique()

    if set(valores_unicos).issubset(set(['Yes', 'No'])):
        print(f'{column}')

        df_leads[column] = df_leads[column].apply((lambda x: 1 if x == 'Yes' else 0))

Do Not Email
Do Not Call
Search
Newspaper Article
X Education Forums
Newspaper
Digital Advertisement
Through Recommendations
A free copy of Mastering The Interview


/tmp/ipykernel_15765/420006417.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_leads.select_dtypes(include='object'):


In [89]:
colunas_categoricas = df_leads.select_dtypes(include='object').columns
df_leads.dropna(subset=colunas_categoricas, inplace=True)

/tmp/ipykernel_15765/2161987220.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = df_leads.select_dtypes(include='object').columns


In [93]:
for column in df_leads.select_dtypes(include='number'):
    contagem_nulas = df_leads[column].isnull().sum()

    if contagem_nulas / len(df_leads) * 100 > 25:
        print(f'{column}: {contagem_nulas / len(df_leads) * 100:.2f}%')

        df_leads.drop(column, axis=1, inplace=True)

Asymmetrique Activity Score: 45.69%
Asymmetrique Profile Score: 45.69%


In [94]:
colunas_numericas = df_leads.select_dtypes(include='number').columns
df_leads.dropna(subset=colunas_numericas, inplace=True)

### EDA

In [98]:
df_leads.describe()

,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview
count,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000
mean,0.078907,0.000220,0.378554,3.456028,482.887481,2.370151,0.001543,0.000220,0.000110,0.000110,0.000441,0.000771,0.318272
std,0.269608,0.014845,0.485053,4.858802,545.256560,2.160871,0.039251,0.014845,0.010498,0.010498,0.020992,0.027766,0.465831
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,3.000000,246.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,1.000000,5.000000,922.750000,3.200000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,251.000000,2272.000000,55.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
